# Autoencoder Feature Weightages

This notebook trains an autoencoder on selected Excel metrics and outputs normalized feature weightages as JSON.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
excel_path = Path("YOUR_EXCEL_PATH.xlsx")
metrics = ["metric_a", "metric_b", "metric_c"]  # update with desired columns
output_path = Path("weightages.json")
latent_dim = 4
epochs = 200
batch_size = 32
learning_rate = 1e-3
seed = 42
device = "cpu"
validation_split = 0.2
test_split = 0.1
patience = 20
min_delta = 1e-4
num_seeds = 3
permutation_repeats = 20
weight_decay = 1e-4
grad_clip = 1.0


In [ ]:
def validate_splits(validation_split: float, test_split: float) -> None:
    if not 0.0 < validation_split < 1.0:
        raise ValueError("validation_split must be between 0 and 1 (exclusive).")
    if not 0.0 < test_split < 1.0:
        raise ValueError("test_split must be between 0 and 1 (exclusive).")
    if validation_split + test_split >= 1.0:
        raise ValueError("validation_split + test_split must be less than 1.")


def split_dataframe(data: pd.DataFrame, validation_split: float, test_split: float, seed: int):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(data))
    val_size = int(len(data) * validation_split)
    test_size = int(len(data) * test_split)
    train_size = len(data) - val_size - test_size
    train_idx = indices[:train_size]
    val_idx = indices[train_size : train_size + val_size]
    test_idx = indices[train_size + val_size :]
    return (
        data.iloc[train_idx].reset_index(drop=True),
        data.iloc[val_idx].reset_index(drop=True),
        data.iloc[test_idx].reset_index(drop=True),
    )


def replace_sentinel_values(data: pd.DataFrame) -> pd.DataFrame:
    sentinel_values = {2147483647, 9.223372e18}
    return data.replace(list(sentinel_values), np.nan)


def impute_missing(train_data: pd.DataFrame, eval_data: pd.DataFrame | None):
    medians = train_data.median()
    train_imputed = train_data.fillna(medians)
    eval_imputed = eval_data.fillna(medians) if eval_data is not None else None
    return train_imputed, eval_imputed


def add_missing_flags(train_data: pd.DataFrame, eval_data: pd.DataFrame | None, threshold: float = 0.05):
    missing_flags = []
    train_missing = train_data.isna().mean()
    flagged = train_missing[train_missing > threshold].index
    for column in flagged:
        flag_name = f"{column}__missing"
        missing_flags.append(flag_name)
        train_data[flag_name] = train_data[column].isna().astype(float)
        if eval_data is not None:
            eval_data[flag_name] = eval_data[column].isna().astype(float)
    return train_data, eval_data


def apply_column_transforms(train_data: pd.DataFrame, eval_data: pd.DataFrame | None):
    transformers = {}
    log_columns = {
        "freeStorageSpaceInBytes",
        "readIOPS",
        "writeIOPS",
        "tcpDataReceivedMB",
        "tcpDataSentMB",
    }
    for column in train_data.columns:
        if column.endswith("__missing"):
            transformers[column] = None
            continue
        series = train_data[column]
        if column in log_columns:
            train_data[column] = np.log1p(series.clip(lower=0))
            if eval_data is not None:
                eval_data[column] = np.log1p(eval_data[column].clip(lower=0))
            transformers[column] = None
            continue
        skewness = series.skew()
        if abs(skewness) > 1.0:
            transformer = PowerTransformer(method="yeo-johnson", standardize=False)
            train_data[column] = transformer.fit_transform(series.to_frame()).ravel()
            if eval_data is not None:
                eval_data[column] = transformer.transform(eval_data[column].to_frame()).ravel()
            transformers[column] = transformer
        else:
            transformers[column] = None
    return train_data, eval_data, transformers


def has_heavy_outliers(data: pd.DataFrame) -> bool:
    quantiles = data.quantile([0.25, 0.75])
    iqr = quantiles.loc[0.75] - quantiles.loc[0.25]
    if (iqr == 0).all():
        return False
    high_outliers = (data > (quantiles.loc[0.75] + 3 * iqr)).any()
    return bool(high_outliers.any())


def prepare_features(train_data: pd.DataFrame, eval_data: pd.DataFrame | None = None):
    numeric_train = train_data.apply(pd.to_numeric, errors="coerce")
    numeric_eval = eval_data.apply(pd.to_numeric, errors="coerce") if eval_data is not None else None
    numeric_train = replace_sentinel_values(numeric_train)
    if numeric_eval is not None:
        numeric_eval = replace_sentinel_values(numeric_eval)
    numeric_train, numeric_eval = add_missing_flags(numeric_train, numeric_eval)
    numeric_train, numeric_eval = impute_missing(numeric_train, numeric_eval)
    numeric_train, numeric_eval, transformers = apply_column_transforms(numeric_train, numeric_eval)
    if numeric_train.empty:
        raise ValueError("No numeric rows available after cleaning the data.")
    scaler = RobustScaler() if has_heavy_outliers(numeric_train) else StandardScaler()
    scaled_train = scaler.fit_transform(numeric_train.values)
    scaled_eval = scaler.transform(numeric_eval.values) if numeric_eval is not None else None
    return scaled_train.astype(np.float32), scaled_eval.astype(np.float32) if scaled_eval is not None else None, scaler, transformers, list(numeric_train.columns)


In [ ]:
data = pd.read_excel(excel_path)
if metrics:
    missing = [metric for metric in metrics if metric not in data.columns]
    if missing:
        raise ValueError("Metrics not found in Excel data: " + ", ".join(sorted(missing)))
    data = data[list(metrics)]

validate_splits(validation_split, test_split)
train_data, val_data, test_data = split_dataframe(data, validation_split, test_split, seed)
train_features, val_features, scaler, transformers, feature_names = prepare_features(train_data, val_data)
_, test_features, _, _, _ = prepare_features(train_data, test_data)
if train_features is None or val_features is None or test_features is None:
    raise ValueError("Failed to build training/validation/test feature sets.")
if latent_dim not in {2, 4, 8}:
    raise ValueError("latent_dim must be one of 2, 4, or 8.")
if latent_dim >= train_features.shape[1]:
    raise ValueError("latent_dim must be smaller than the number of features.")
batch_size = min(batch_size, max(1, train_features.shape[0]))
num_seeds = max(1, min(num_seeds, 5))


In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        dropout = 0.1
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.LayerNorm(16),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(16, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(32, 16),
            nn.LayerNorm(16),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.LayerNorm(16),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(16, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(32, 16),
            nn.LayerNorm(16),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(16, input_dim),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        latent = self.encoder(inputs)
        return self.decoder(latent)


def reconstruction_loss(model: AutoEncoder, features: np.ndarray, loss_fn: nn.Module, device: str) -> float:
    dataset = TensorDataset(torch.from_numpy(features).float())
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    losses = []
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            reconstruction = model(batch)
            losses.append(loss_fn(reconstruction, batch).item())
    return float(np.mean(losses)) if losses else float("inf")


def train_autoencoder(train_features: np.ndarray, val_features: np.ndarray, test_features: np.ndarray, run_seed: int):
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    model = AutoEncoder(input_dim=train_features.shape[1], latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    train_dataset = TensorDataset(torch.from_numpy(train_features).float())
    val_dataset = TensorDataset(torch.from_numpy(val_features).float())
    test_dataset = TensorDataset(torch.from_numpy(test_features).float())
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val = float("inf")
    epochs_without_improve = 0

    for _ in range(epochs):
        model.train()
        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            reconstruction = model(batch)
            loss = loss_fn(reconstruction, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                reconstruction = model(batch)
                val_losses.append(loss_fn(reconstruction, batch).item())
        val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
        scheduler.step(val_loss)

        if best_val - val_loss > min_delta:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    test_losses = []
    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)
            reconstruction = model(batch)
            test_losses.append(loss_fn(reconstruction, batch).item())
    test_loss = float(np.mean(test_losses)) if test_losses else float("inf")

    return model, {"validation_loss": best_val, "test_loss": test_loss}


def normalize_importances(importances: np.ndarray, names: list[str]):
    total = float(np.sum(importances))
    if math.isclose(total, 0.0):
        normalized = np.zeros_like(importances)
    else:
        normalized = importances / total
    paired = [(feature, float(weight)) for feature, weight in zip(names, normalized, strict=False)]
    paired.sort(key=lambda item: item[1], reverse=True)
    return dict(paired)


def compute_permutation_importance(model: AutoEncoder, features: np.ndarray, names: list[str], run_seed: int):
    model.eval()
    loss_fn = nn.SmoothL1Loss()
    rng = np.random.default_rng(run_seed)
    base_loss = reconstruction_loss(model, features, loss_fn, device)
    importances = np.zeros(features.shape[1], dtype=np.float64)
    importances_std = np.zeros(features.shape[1], dtype=np.float64)
    for idx in range(features.shape[1]):
        losses = []
        for _ in range(permutation_repeats):
            shuffled = features.copy()
            rng.shuffle(shuffled[:, idx])
            loss = reconstruction_loss(model, shuffled, loss_fn, device)
            losses.append(loss - base_loss)
        importances[idx] = float(np.mean(losses))
        importances_std[idx] = float(np.std(losses))
    return normalize_importances(importances, names), normalize_importances(importances_std, names)


def build_feature_groups(feature_names: list[str]):
    groups = {}
    zscaler_indices = [idx for idx, name in enumerate(feature_names) if "zscaler" in name.lower()]
    if zscaler_indices:
        groups["zscaler"] = zscaler_indices
    iops_indices = [idx for idx, name in enumerate(feature_names) if name in {"readIOPS", "writeIOPS"}]
    if iops_indices:
        groups["iops"] = iops_indices
    tcp_indices = [idx for idx, name in enumerate(feature_names) if name in {"tcpDataReceivedMB", "tcpDataSentMB"}]
    if tcp_indices:
        groups["tcp"] = tcp_indices
    return groups


def compute_group_permutation_importance(model: AutoEncoder, features: np.ndarray, groups: dict[str, list[int]], run_seed: int):
    model.eval()
    loss_fn = nn.SmoothL1Loss()
    rng = np.random.default_rng(run_seed)
    base_loss = reconstruction_loss(model, features, loss_fn, device)
    group_names = list(groups.keys())
    importances = np.zeros(len(group_names), dtype=np.float64)
    importances_std = np.zeros(len(group_names), dtype=np.float64)
    for idx, name in enumerate(group_names):
        losses = []
        group_indices = groups[name]
        for _ in range(permutation_repeats):
            shuffled = features.copy()
            for column_idx in group_indices:
                rng.shuffle(shuffled[:, column_idx])
            loss = reconstruction_loss(model, shuffled, loss_fn, device)
            losses.append(loss - base_loss)
        importances[idx] = float(np.mean(losses))
        importances_std[idx] = float(np.std(losses))
    return normalize_importances(importances, group_names), normalize_importances(importances_std, group_names)


def summarize_importances(importances: list[dict[str, float]], names: list[str]):
    matrix = np.array([[importance.get(name, 0.0) for name in names] for importance in importances])
    mean_vals = matrix.mean(axis=0)
    std_vals = matrix.std(axis=0)
    return normalize_importances(mean_vals, names), normalize_importances(std_vals, names)


def top_k_overlap(weightage_sets: list[dict[str, float]], k: int = 5) -> float:
    if not weightage_sets:
        return 0.0
    top_sets = [set(list(weights.keys())[:k]) for weights in weightage_sets if weights]
    if not top_sets:
        return 0.0
    intersection = set.intersection(*top_sets)
    return len(intersection) / k


In [ ]:
weightage_runs = []
group_weightage_runs = []
groups = build_feature_groups(feature_names)

for offset in range(num_seeds):
    run_seed = seed + offset
    model, metrics = train_autoencoder(train_features, val_features, test_features, run_seed)
    weightages, _ = compute_permutation_importance(model, val_features, feature_names, run_seed)
    weightage_runs.append(weightages)
    if groups:
        group_weightages, _ = compute_group_permutation_importance(model, val_features, groups, run_seed)
        group_weightage_runs.append(group_weightages)

weightages_mean, weightages_std = summarize_importances(weightage_runs, feature_names)
group_mean = {}
group_std = {}
if group_weightage_runs and groups:
    group_names = list(groups.keys())
    group_mean, group_std = summarize_importances(group_weightage_runs, group_names)

top5_overlap = top_k_overlap(weightage_runs, k=5)


In [ ]:
output_payload = {
    "weightages_mean": weightages_mean,
    "weightages_std": weightages_std,
    "group_weightages_mean": group_mean,
    "group_weightages_std": group_std,
    "top5_overlap": top5_overlap,
}
output_path.write_text(json.dumps(output_payload, indent=2), encoding="utf-8")
output_path
